# **TRAINING MODEL**

In [37]:
import os
import csv
from datetime import datetime
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, f1_score, log_loss
from sklearn.model_selection import RepeatedStratifiedKFold, train_test_split
from sklearn.tree import DecisionTreeClassifier
import case_config as cfg

# ============================================================
# TODO: FILES CONFIGURATION 
# ============================================================

TRAIN_FILE = os.path.join("data", "train.csv")
MODEL_FILE = r"model.joblib" # model output 

TRAIN_SIZE = 0.8
LOG_FILE = r"team3B_training_submit.csv"

# ============================================================
# NOTE: HYPERPARAMETERS TUING
# ============================================================
# Set AUTO_TUNE = True on a brand-new dataset if you want CV search
# across tree sizes + light pruning; False uses the proven default.
AUTO_TUNE = False

CRITERION = "entropy"
MAX_DEPTH = 8
MIN_SAMPLES_SPLIT = 18
MIN_SAMPLES_LEAF = 2
RANDOM_STATE = 42
CCP_ALPHA = 0.006

# ============================================================
# LOAD DATA
# ============================================================
def load_labelled(path):
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()
    required = cfg.FEATURES + ["y"]
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f"Missing required column(s) in {path}: {missing}")

    raw_len = len(df)
    if "timestamp" in df.columns:
        df = df.drop_duplicates(subset=["timestamp"], keep="last")
    for col in required:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=required).copy()
    df["y"] = df["y"].astype(int)
    df = df[df["y"].isin(cfg.class_ids())].copy()

    dropped = raw_len - len(df)
    if dropped > 0:
        print(f"Cleaned {dropped} invalid/duplicate row(s) from {path}")

    x = df[cfg.FEATURES].to_numpy(dtype=float)
    y = df["y"].to_numpy(dtype=int)
    return df, x, y

def show_class_balance(df, title):
    print("")
    print("===== " + title + " =====")
    counts = df["y"].value_counts().sort_index()
    for class_id, n in counts.items():
        print(
            "  %d %-10s %5d  %5.1f%%"
            % (
                class_id,
                cfg.label_of(class_id),
                n,
                100.0 * n / len(df)
            )
        )

# ============================================================
# BUILD & AUTO-TUNE MODEL
# ============================================================

def default_params():
    return {
        "criterion": CRITERION,
        "max_depth": MAX_DEPTH,
        "min_samples_split": MIN_SAMPLES_SPLIT,
        "min_samples_leaf": MIN_SAMPLES_LEAF,
        "ccp_alpha": CCP_ALPHA,
        "random_state": RANDOM_STATE,
    }

def build_model(params=None):
    p = default_params() if params is None else params
    return DecisionTreeClassifier(
        criterion=p["criterion"],
        max_depth=p["max_depth"],
        min_samples_split=p["min_samples_split"],
        min_samples_leaf=p["min_samples_leaf"],
        ccp_alpha=p.get("ccp_alpha", 0.0),
        random_state=p["random_state"],
    )

def auto_tune_params(x, y, random_state=RANDOM_STATE):
    counts = np.bincount(y)
    min_class = int(counts[counts > 0].min())
    n_splits = max(2, min(5, min_class))
    n_repeats = 4 if len(x) <= 1500 else 2
    rskf = RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=random_state)
    splits = list(rskf.split(x, y))
    labels = cfg.class_ids()

    n = len(x)
    s_small = max(4, min(10, n // 60))
    s_med = max(10, min(18, n // 35))
    s_large = max(16, min(28, n // 22))
    candidates = []
    for crit in ["gini", "entropy"]:
        for md in [6, 8, 10, None]:
            for mss, msl in [(2, 2), (s_small, 2), (s_med, 2), (s_med, 4), (s_large, 3), (s_large, 5), (2, 5), (2, 8)]:
                if mss != 2 and mss < 2 * msl:
                    continue
                candidates.append((crit, md, mss, msl))
    candidates = list(dict.fromkeys(candidates))

    stage1 = []
    for crit, md, mss, msl in candidates:
        f1s = []
        for tr_i, va_i in splits:
            clf = DecisionTreeClassifier(
                criterion=crit, max_depth=md,
                min_samples_split=mss, min_samples_leaf=msl,
                random_state=random_state,
            ).fit(x[tr_i], y[tr_i])
            f1s.append(f1_score(y[va_i], clf.predict(x[va_i]), average="macro", labels=labels, zero_division=0))
        stage1.append((float(np.mean(f1s)), float(np.std(f1s)), crit, md, mss, msl))
    stage1.sort(key=lambda r: (r[0], -r[1]), reverse=True)

    stage2 = []
    for _, _, crit, md, mss, msl in stage1[:6]:
        for alpha in [0.0, 0.002, 0.004, 0.006]:
            f1s, lls = [], []
            for tr_i, va_i in splits:
                clf = DecisionTreeClassifier(
                    criterion=crit, max_depth=md,
                    min_samples_split=mss, min_samples_leaf=msl,
                    ccp_alpha=alpha, random_state=random_state,
                ).fit(x[tr_i], y[tr_i])
                pred = clf.predict(x[va_i])
                prob = clf.predict_proba(x[va_i])
                f1s.append(f1_score(y[va_i], pred, average="macro", labels=labels, zero_division=0))
                if prob.shape[1] == len(labels):
                    lls.append(log_loss(y[va_i], prob, labels=labels))
            avg_ll = float(np.mean(lls)) if lls else 0.0
            stage2.append((float(np.mean(f1s)), float(np.std(f1s)), avg_ll, crit, md, mss, msl, alpha))

    stage2.sort(key=lambda r: (r[0], -r[2], -r[1]), reverse=True)
    print("\n===== AUTO-TUNE TOP 5 (Repeated Stratified CV) =====")
    for r in stage2[:5]:
        print(
            f"  CV={r[0]:.4f}+/-{r[1]:.4f} LL={r[2]:.3f} | "
            f"{r[3]:7s} d={str(r[4]):4s} s={r[5]:2d} l={r[6]:2d} ccp={r[7]:.3f}"
        )

    max_f1 = max(r[0] for r in stage2)
    near_best = [r for r in stage2 if r[0] >= max_f1 - 0.0005]
    near_best.sort(key=lambda r: (r[2], r[1]))
    w = near_best[0]
    return {
        "criterion": w[3],
        "max_depth": w[4],
        "min_samples_split": w[5],
        "min_samples_leaf": w[6],
        "ccp_alpha": w[7],
        "random_state": random_state,
    }
    
# ============================================================
# EVALUATE
# ============================================================

def evaluate(model, x, y, title):
    predicted = model.predict(x)
    score = f1_score(
        y,
        predicted,
        average="macro",
        labels=cfg.class_ids()
    )
    print("")
    print("===== " + title + " =====")
    print("Macro-F1: %.4f" % score)
    print(
        classification_report(
            y,
            predicted,
            labels=cfg.class_ids(),
            target_names=cfg.CLASS_LABELS,
            digits=4,
            zero_division=0
        )
    )
    print("Confusion matrix (rows = true, columns = predicted):")
    print(
        confusion_matrix(
            y,
            predicted,
            labels=cfg.class_ids()
        )
    )
    return score

def save_log(timestamp, criterion, max_depth, min_samples_split, min_samples_leaf, random_state, val_f1):
    file_exists = os.path.isfile(LOG_FILE)
    
    headers = [
        "timestamp", 
        "criterion", 
        "max_depth", 
        "min_samples_split", 
        "min_samples_leaf",
        "random_state", 
        "val_macro_f1"
    ]
    
    data = [
        timestamp, 
        criterion, 
        max_depth, 
        min_samples_split, 
        min_samples_leaf,
        random_state, 
        round(val_f1, 4)
    ]
    
    with open(LOG_FILE, mode="a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(headers)
        writer.writerow(data)
def main():
    print("Case:  " + cfg.CASE_TITLE)
    print("Model: " + cfg.MODEL_NAME)

    # NOTE: Load Train.csv
    train_df, x, y = load_labelled(TRAIN_FILE)
    print("")
    print("Total rows in train.csv: %d" % len(train_df))
    show_class_balance(train_df, "CLASS BALANCE: TRAINING")

    # NOTE: Split train.csv into TRAIN_SIZE train / validation
    counts = np.bincount(y)
    min_count = int(counts[counts > 0].min())
    strat = y if len(np.unique(y)) > 1 and min_count >= 2 else None
    if strat is None:
        print("WARNING: Too few samples in at least one class for stratified split; using random split.")

    x_train, x_val, y_train, y_val = train_test_split(
        x, y, train_size=TRAIN_SIZE, stratify=strat, random_state=RANDOM_STATE
    )

    print("Training rows:   %d" % len(x_train))
    print("Validation rows: %d" % len(x_val))

    chosen_params = auto_tune_params(x_train, y_train, random_state=RANDOM_STATE) if AUTO_TUNE else default_params()
    print("Using hyperparameters:", chosen_params)

    # NOTE: Train model using the training part
    model = build_model(chosen_params)
    model.fit(x_train, y_train)
    print("Model trained on %d%% training subset." % round(TRAIN_SIZE * 100))
    print(model)

    # NOTE: Evaluate using the validation part
    validation_f1 = evaluate(model, x_val, y_val, "VALIDATION SET")

    # NOTE: Final model using ALL train.csv
    final_model = build_model(chosen_params)
    final_model.fit(x, y)
    print("===== FINAL MODEL TRAINING =====")
    print("Final model trained on all %d rows in train.csv." % len(x))

    # NOTE: Save Final model
    joblib.dump(final_model, MODEL_FILE)
    print("Saved model: " + MODEL_FILE)
    print("Validation Macro-F1: %.4f" % validation_f1)

    current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    save_log(
        timestamp=current_time,
        criterion=chosen_params["criterion"],
        max_depth=chosen_params["max_depth"],
        min_samples_split=chosen_params["min_samples_split"],
        min_samples_leaf=chosen_params["min_samples_leaf"],
        random_state=chosen_params["random_state"],
        val_f1=validation_f1
    )
    print("Log saved to: " + LOG_FILE)
if __name__ == "__main__":
    main()

Case:  Greenhouse Environment State
Model: decision_tree

Total rows in train.csv: 613

===== CLASS BALANCE: TRAINING =====
  0 normal       378   61.7%
  1 alert        155   25.3%
  2 alarm         80   13.1%
Training rows:   490
Validation rows: 123
Using hyperparameters: {'criterion': 'entropy', 'max_depth': 8, 'min_samples_split': 18, 'min_samples_leaf': 2, 'ccp_alpha': 0.006, 'random_state': 42}
Model trained on 80% training subset.
DecisionTreeClassifier(ccp_alpha=0.006, criterion='entropy', max_depth=8,
                       min_samples_leaf=2, min_samples_split=18,
                       random_state=42)

===== VALIDATION SET =====
Macro-F1: 0.8770
              precision    recall  f1-score   support

      normal     0.9157    1.0000    0.9560        76
       alert     0.9167    0.7097    0.8000        31
       alarm     0.8750    0.8750    0.8750        16

    accuracy                         0.9106       123
   macro avg     0.9024    0.8616    0.8770       123
weighte

# **TESTING MODEL**

In [38]:
import os
import csv
from datetime import datetime
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import case_config as cfg

# ============================================================
# TODO: FILES CONFIGURATION 
# ============================================================
TEST_FILE = os.path.join("data", "test.csv")
MODEL_FILE = r"model.joblib" # same file cell 1 saves
LOG_FILE = r"team3B_testing_submit.csv"  

# ============================================================
# LOAD DATA
# ============================================================

def load_labelled(path):
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()
    required = cfg.FEATURES + ["y"]
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f"Missing required column(s) in {path}: {missing}")

    raw_len = len(df)
    if "timestamp" in df.columns:
        df = df.drop_duplicates(subset=["timestamp"], keep="last")
    for col in required:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=required).copy()
    df["y"] = df["y"].astype(int)
    df = df[df["y"].isin(cfg.class_ids())].copy()

    dropped = raw_len - len(df)
    if dropped > 0:
        print(f"Cleaned {dropped} invalid/duplicate row(s) from {path}")

    x = df[cfg.FEATURES].to_numpy(dtype=float)
    y = df["y"].to_numpy(dtype=int)
    return df, x, y

def show_class_balance(df):
    print("")
    print("===== CLASS BALANCE: UNSEEN TEST SET =====")
    counts = df["y"].value_counts().sort_index()
    for class_id, n in counts.items():
        print(
            "  %d %-10s %5d  %5.1f%%"
            % (
                class_id,
                cfg.label_of(class_id),
                n,
                100.0 * n / len(df)
            )
        )

# ============================================================
# EVALUATE
# ============================================================

def evaluate(model, x, y):
    predicted = model.predict(x)
    score = f1_score(
        y,
        predicted,
        average="macro",
        labels=cfg.class_ids()
    )
    print("")
    print("===== FINAL UNSEEN TEST RESULT =====")
    print("Macro-F1: %.4f" % score)
    print(
        classification_report(
            y,
            predicted,
            labels=cfg.class_ids(),
            target_names=cfg.CLASS_LABELS,
            digits=4,
            zero_division=0
        )
    )
    print("Confusion matrix (rows = true, columns = predicted):")
    print(
        confusion_matrix(
            y,
            predicted,
            labels=cfg.class_ids()
        )
    )
    return score
# ============================================================
# FUNCTION FOR LOGGING TEST RESULTS
# ============================================================
def save_test_log(timestamp, model_file, params, test_f1):
    file_exists = os.path.isfile(LOG_FILE)
    
    headers = [
        "timestamp", 
        "model_file",
        "criterion", 
        "max_depth", 
        "min_samples_split", 
        "min_samples_leaf",
        "random_state", 
        "test_macro_f1"
    ]
    
    data = [
        timestamp,
        model_file,
        params.get("criterion"),
        params.get("max_depth"),
        params.get("min_samples_split"),
        params.get("min_samples_leaf"),
        params.get("random_state"),
        round(test_f1, 4)
    ]
    
    with open(LOG_FILE, mode="a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(headers)
        writer.writerow(data)
def main():
    print("Case:  " + cfg.CASE_TITLE)
    print("Model: " + cfg.MODEL_NAME)

    # NOTE: Load trained model
    model = joblib.load(MODEL_FILE)
    print("Loaded model: " + MODEL_FILE)

    # NOTE: Load test dataset
    test_df, x_test, y_test = load_labelled(TEST_FILE)
    print("Unseen test rows: %d" % len(test_df))
    show_class_balance(test_df)

    # NOTE: Final evaluation
    final_f1 = evaluate(
        model,
        x_test,
        y_test
    )
    print("Final unseen-test Macro-F1: %.4f" % final_f1)
    current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    params = model.get_params() 
    
    save_test_log(
        timestamp=current_time,
        model_file=MODEL_FILE,
        params=params,
        test_f1=final_f1
    )
    print("Test Log saved to: " + LOG_FILE)

if __name__ == "__main__":
    main()

Case:  Greenhouse Environment State
Model: decision_tree
Loaded model: model.joblib
Unseen test rows: 500

===== CLASS BALANCE: UNSEEN TEST SET =====
  0 normal       299   59.8%
  1 alert        118   23.6%
  2 alarm         83   16.6%

===== FINAL UNSEEN TEST RESULT =====
Macro-F1: 0.9106
              precision    recall  f1-score   support

      normal     0.9508    0.9699    0.9603       299
       alert     0.8596    0.8305    0.8448       118
       alarm     0.9383    0.9157    0.9268        83

    accuracy                         0.9280       500
   macro avg     0.9162    0.9054    0.9106       500
weighted avg     0.9272    0.9280    0.9275       500

Confusion matrix (rows = true, columns = predicted):
[[290   9   0]
 [ 15  98   5]
 [  0   7  76]]
Final unseen-test Macro-F1: 0.9106
Test Log saved to: team3B_testing_submit.csv
